# 01 · Extracción de texto (PDF → texto plano)

<a href="https://colab.research.google.com/github/manuelarguelles/tyv-demo-colab/blob/main/notebooks/01_extraccion_texto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Primer paso del pipeline de filtrado curricular de **Terry & Valdez**: antes de
que cualquier modelo de lenguaje vea un currículum, el sistema necesita
convertirlo de PDF a **texto plano**. Este notebook usa un **PDF real que vos
subís** — no un CV fabricado — y lo procesa con la misma herramienta que usa
el sistema real: **Poppler** (`pdftotext`), un extractor de código abierto.

**Por qué no usar una librería Python "pura" (como `PyPDF2`) en su lugar:**
`pdftotext -layout` conserva el orden espacial del texto en la página — dos
columnas, tablas, secciones alineadas — que es exactamente lo que hace un CV
legible tanto para una persona como para un modelo de lenguaje. Extraer sin
`-layout` puede entremezclar columnas y romper el orden de lectura.

> El PDF que subas queda **solo en esta sesión de Colab** (se borra al
> cerrarla) — nunca se guarda en este repositorio.


## 1. Instalar Poppler (el extractor real)

In [ ]:
!apt-get -qq update && apt-get -qq install -y poppler-utils > /dev/null
!pdftotext -v


## 2. Subir un CV real (PDF)

In [ ]:
def cargar_pdf() -> str:
    """Pide un PDF real al usuario. En Colab, abre el selector de archivos
    del navegador — el archivo se sube a la sesión y NO queda guardado en
    este repositorio. Corriendo localmente (fuera de Colab), busca un PDF
    ya copiado a `materiales/cvs/` (ver materiales/README.md)."""
    try:
        from google.colab import files
        print("Subí el PDF de un CV real (queda solo en esta sesión de Colab).")
        subido = files.upload()
        if not subido:
            raise RuntimeError("No se subió ningún archivo.")
        return next(iter(subido))
    except ImportError:
        import glob
        candidatos = sorted(glob.glob("materiales/cvs/*.pdf"))
        if not candidatos:
            raise FileNotFoundError(
                "Corriendo fuera de Colab: copiá un PDF real a materiales/cvs/ "
                "(ver materiales/README.md) y volvé a correr esta celda."
            )
        print(f"Usando el primer PDF encontrado en materiales/cvs/: {candidatos[0]}")
        return candidatos[0]

ruta_pdf = cargar_pdf()
print(f"\nArchivo listo: {ruta_pdf}")


## 3. Extraer el texto con `pdftotext -layout`

Esta es la llamada real que hace el sistema (vía `subprocess`).

In [ ]:
import subprocess

def extraer_texto_pdf(ruta_pdf: str) -> str:
    """Equivalente a la función `extraer()` del servidor real:
    `pdftotext -layout <pdf> -` conserva el orden espacial del texto."""
    resultado = subprocess.run(
        ["pdftotext", "-layout", ruta_pdf, "-"],
        capture_output=True, text=True, check=True,
    )
    return resultado.stdout

MAX_CV = 12_000  # caracteres — mismo límite que usa el sistema real (p90 sobre 287 CVs)

def recortar_a_limite(texto: str, limite: int = MAX_CV) -> str:
    if len(texto) <= limite:
        return texto
    return texto[:limite] + "\n[... recortado: documento más largo que el límite operativo ...]"

texto_extraido = extraer_texto_pdf(ruta_pdf)
print(texto_extraido)


## 4. Por qué `-layout` importa (comparación)

Repetimos la extracción sin `-layout` sobre el mismo PDF. Con un CV a una
sola columna el efecto puede ser sutil; con un CV real de dos columnas o con
tablas (fechas a la derecha, cargos a la izquierda), la diferencia es la que
separa un texto legible de un texto con palabras mezcladas fuera de orden.

In [ ]:
resultado_sin_layout = subprocess.run(
    ["pdftotext", ruta_pdf, "-"],
    capture_output=True, text=True, check=True,
).stdout

print("── CON -layout (primeras 8 líneas) ──")
print("\n".join(texto_extraido.splitlines()[:8]))
print()
print("── SIN -layout (primeras 8 líneas) ──")
print("\n".join(resultado_sin_layout.splitlines()[:8]))


## 5. Límite de tamaño

El sistema real recorta el CV a un máximo de caracteres antes de enviarlo al modelo (calibrado sobre 287 CVs reales del proyecto: percentil 90 ≈ 13 600 caracteres, ~3 páginas).

In [ ]:
texto_final = recortar_a_limite(texto_extraido)
print(f"Longitud del CV extraído: {len(texto_extraido)} caracteres (límite: {MAX_CV})")


## Siguiente paso

`texto_final` todavía contiene los datos personales del candidato real que
subiste — nombre, correo, teléfono, DNI, fecha de nacimiento, dirección.
Antes de que cualquier modelo de IA lo vea, pasa por la etapa de
**anonimización** → `02_anonimizacion.ipynb`.

---
*Este material es contenido educativo de apoyo a una tesis de maestría (Terry & Valdez — sistema de filtrado curricular). El PDF que subís y el nombre que ingresás quedan solo en la memoria de esta sesión de Colab — nunca se guardan en este repositorio ni se envían a ningún lado salvo, si activás el modo real, al proveedor del modelo (DeepSeek), y solo el texto ya anonimizado. Ver `materiales/README.md` para trabajar con archivos reales en disco de forma local.*
